# Chunking under concurrent load
Compare paragraph and sentence streaming through one warmed FIFO worker with bursts of 1, 2, and 4 requests. Listen to the paired clips and download the raw timings.

In [ ]:
import os
import shutil
import subprocess
import tempfile

smi = shutil.which("nvidia-smi")
if not smi:
    raise RuntimeError("No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell.")
subprocess.run([smi, "-L"], check=True)

%pip -q install uv
os.environ["VOICEFORGE_PHASE7_RESULTS"] = tempfile.mkdtemp(prefix="voiceforge-phase7-", dir="/content")

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
if ! command -v espeak-ng >/dev/null; then
  apt-get -qq update
  apt-get -qq install -y espeak-ng
fi
repo=/content/voiceforge-chunking-load
if [[ ! -d "$repo/.git" ]]; then
  git clone --branch voiceforge-gpu-precision https://github.com/dne-labs/Voice-inference.git "$repo"
fi
cd "$repo/services/inference"
uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  --with 'en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl' \
  voiceforge-synthesize --device cuda --output /tmp/voiceforge-phase7-warmup.wav

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
cd /content/voiceforge-chunking-load/services/inference
HF_HUB_OFFLINE=1 OMP_NUM_THREADS=2 uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  --with 'en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl' \
  python ../../experiments/06_adaptive_chunking/load.py --output "$VOICEFORGE_PHASE7_RESULTS"

In [ ]:
from pathlib import Path
import json
import os
import shutil
from google.colab import files

results = Path(os.environ["VOICEFORGE_PHASE7_RESULTS"])
from IPython.display import Audio, display
for policy in ("paragraph", "sentence"):
    print(policy)
    display(Audio(filename=str(results / f"{policy}.wav")))
print("Saved chunking-load.json")
files.download(shutil.make_archive(str(results), "zip", results))
